# Phase 7 on Colab: P7.5 training runs (+ hinge 1.5 seeds 3-4)

**How to run (VS Code):** *Select Kernel* -> **Colab** -> **New Colab Server** -> **T4 GPU**, then **Run All**.
Cell 3 asks for the watermark key in a hidden prompt. Copy it on the laptop first (PowerShell, repo root; it goes to
the clipboard and is never printed):

```powershell
venv\Scripts\python.exe -c "print(open('keys/watermark_key.bin','rb').read().hex(), end='')" | Set-Clipboard
```

| Part | What | Runs | Time on a T4 |
|---|---|---|---|
| A | P7.5 group `seeds` as planned (BCE): baseline seeds 3, 4 and rs32 lambda 20 seeds 3, 4 -> 5 paired seeds | 4 | ~35 min |
| B | hinge 1.5 seeds 3, 4, so the hinge design also has 5 paired seeds if it is adopted | 2 | ~17 min |
| C | rest of P7.5 (`ecc_off`, `lambda`, `payload`, BCE). **OFF** until the hinge 1.5 attack results decide the loss | 17 | ~2.4 h |

At the end the last cell downloads `colab_results.zip`. Put it in the repo root on the laptop; Claude unpacks and
verifies it. Finished runs are skipped, so after a timeout just reconnect and Run All again.

In [ ]:
# 1. GPU check: stop early if Colab gave a CPU-only runtime
import sys
import torch
print(sys.version.split()[0], "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU. Reconnect: Select Kernel -> Colab -> New Colab Server -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code: branch phase6, without runs/attacks (not needed here, ~200 MB)
import os, pathlib, subprocess

REPO_URL = "https://github.com/shield44-project/post-quantum-cryptography.git"
BRANCH = "phase6"
WORK = "/content/pqw"
START_MARK = pathlib.Path("/content/.pqw_start")   # files newer than this are this session's results


def sh(cmd):
    # run a shell command, stream its output, stop Run All if it fails
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError("command failed (exit %d): %s" % (p.returncode, cmd))


if not os.path.isdir(WORK + "/.git"):
    sh(f"git clone --depth 1 --branch {BRANCH} --filter=blob:none --no-checkout {REPO_URL} {WORK}")
    sh(f"git -C {WORK} sparse-checkout set --no-cone '/*' '!/runs/attacks/'")
    sh(f"git -C {WORK} checkout {BRANCH}")
os.chdir(WORK)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
if not START_MARK.exists():
    START_MARK.touch()
sh("git log --oneline -1")
sh("pip install -q 'reedsolo>=1.7'")
os.environ["CIFAR_ROOT"] = "/content/data"
os.environ["PYTHONUNBUFFERED"] = "1"

In [ ]:
# 3. Watermark key: pasted as hex in a hidden prompt, written only to this runtime, fingerprint checked
from getpass import getpass
from embedding.keys import key_fingerprint

KEY_PATH = pathlib.Path("keys/watermark_key.bin")
EXPECTED = "81b57c4b28884f01"            # the key every model in runs/wm_v1 was trained with

if KEY_PATH.exists() and key_fingerprint(KEY_PATH.read_bytes()) == EXPECTED:
    print("key already on this runtime, fingerprint", EXPECTED, "OK")
else:
    key = bytes.fromhex(getpass("Paste the 64 hex characters of keys/watermark_key.bin (hidden): ").strip())
    fp = key_fingerprint(key)
    if fp != EXPECTED:
        raise ValueError("wrong key: fingerprint %s, expected %s. Nothing was written." % (fp, EXPECTED))
    KEY_PATH.parent.mkdir(exist_ok=True)
    KEY_PATH.write_bytes(key)
    os.chmod(KEY_PATH, 0o600)
    del key
    print("key written, fingerprint", fp, "OK")

In [ ]:
# 4. CIFAR-10 to local disk (fails here, not 20 minutes into training, if the download breaks)
import torchvision
for train in (True, False):
    ds = torchvision.datasets.CIFAR10("/content/data", train=train, download=True)
    print("train" if train else "test", len(ds))

In [ ]:
# 5. Optional Google Drive backup (worth it for Part C). Off by default: in VS Code the sign-in may need the
#    "Colab: Mount Google Drive to Server" command instead of drive.mount().
USE_DRIVE = False
BACKUP = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BACKUP = pathlib.Path("/content/drive/MyDrive/pq-watermark-colab")
        BACKUP.mkdir(parents=True, exist_ok=True)
        print("backups go to", BACKUP)
    except Exception as e:
        print("Drive not mounted (%s). Results stay on this runtime until downloaded." % e)

In [ ]:
# 6. Pack this session's results into /content/colab_results.zip (called after every part)
import shutil, zipfile
ZIP = pathlib.Path("/content/colab_results.zip")


def pack_results():
    t0 = START_MARK.stat().st_mtime
    files = sorted(p for p in pathlib.Path("runs").rglob("*") if p.is_file() and p.stat().st_mtime > t0)
    with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.as_posix())
    run_dirs = sorted({p.parent.as_posix() for p in files if p.name == "final.json"})
    print("%s: %d files, %.1f MB, finished runs:" % (ZIP, len(files), ZIP.stat().st_size / 1e6))
    for d in run_dirs:
        print("   ", d)
    if BACKUP is not None:
        shutil.copy(ZIP, BACKUP / ZIP.name)
        print("copied to", BACKUP / ZIP.name)

In [ ]:
# 7. Part A: P7.5 group 'seeds' (BCE, as planned): baseline seeds 3, 4 + rs32 lambda 20 seeds 3, 4  (~35 min)
sh("python -m experiments.phase7.run_sweeps --only seeds")
pack_results()

In [ ]:
# 8. Part B: hinge 1.5 seeds 3, 4 (same recipe as runs/wm_v1/lam20_n256_rs32_hinge1.5 seeds 0-2)  (~17 min)
for s in (3, 4):
    sh(f"python -m training.train_watermarked --seed {s} --epochs 30 --lam 20 --n-bits 256 --ecc rs32 "
       f"--wm-loss hinge --hinge-margin 1.5 --out-root runs/wm_v1")
pack_results()

In [ ]:
# 9. Part C: the rest of P7.5 (ecc_off, lambda, payload; 17 runs, ~2.4 h). Leave False until the loss is decided.
RUN_PART_C = False
sh("python -m experiments.phase7.run_sweeps --dry-run")
if RUN_PART_C:
    sh("python -m experiments.phase7.run_sweeps --only ecc_off lambda payload")
    pack_results()
else:
    print("Part C skipped (RUN_PART_C = False).")

In [ ]:
# 10. Download the results zip to the laptop
pack_results()
try:
    from google.colab import files
    files.download(str(ZIP))
    print("download started: save colab_results.zip into the repo root on the laptop")
except Exception as e:
    print("files.download did not work here (%s)." % e)
    print("Set USE_DRIVE = True in cell 5, run cells 5, 6 and 10 again, then download the zip from")
    print("Google Drive folder 'pq-watermark-colab'.")